# Camada Silver — Limpeza, Padronização e Regras de Negócio

Este notebook aplica todas as transformações sobre os dados brutos da camada Bronze.
**Nenhuma alteração é feita nas tabelas bronze** — toda a lógica opera sobre leituras.

Transformações aplicadas:
- Renomeação de colunas para português
- Tipagem correta de todas as colunas
- Deduplicação com Window Functions
- Traduções de status (inglês → português)
- Colunas derivadas de tempo de entrega
- Tolerância a falhas com `try_to_timestamp`
- Calendário contínuo de cotação do dólar
- Tabela fato consolidada com valores em BRL e USD
- OPTIMIZE + ZORDER nas tabelas fato

## 1 — Imports e Configurações

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import DoubleType, IntegerType, LongType

CATALOG       = "medalhao"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_SILVER}")

print("Catálogo e schema configurados:", CATALOG, ">", SCHEMA_SILVER)

## 2 — Função Auxiliar de Escrita Silver
Centraliza a gravação das tabelas silver com overwriteSchema habilitado.

In [0]:
def salvar_silver(df, nome_tabela: str):
    """
    Grava um DataFrame como tabela Delta na camada Silver.
    Usa modo overwrite com tolerância a mudanças de schema.
    """
    tabela_completa = f"{CATALOG}.{SCHEMA_SILVER}.{nome_tabela}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_completa)
    )
    count = spark.table(tabela_completa).count()
    print(f"✅ {tabela_completa} → {count:,} registros")

## 3 — silver.dim_consumidores
Origem: `bronze.tb_customers`

Regras:
- Renomeação de colunas para português
- Cidade e estado em UPPER CASE
- Deduplicação: mantém apenas o registro mais recente por `id_consumidor`,
  usando Window Function ordenada de forma decrescente por `timestamp_ingestion`

In [0]:
df_customers_raw = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.tb_customers")

df_customers = df_customers_raw.select(
    F.col("customer_id").alias("id_consumidor"),
    F.col("customer_zip_code_prefix").cast(IntegerType()).alias("prefixo_cep"),
    F.col("customer_name").alias("nome_consumidor"),
    F.upper(F.col("customer_city")).alias("cidade"),    
    F.upper(F.col("customer_state")).alias("estado"),  
    F.col("timestamp_ingestion")
)

window_dedup = Window.partitionBy("id_consumidor").orderBy(F.desc("timestamp_ingestion"))

df_dim_consumidores = (
    df_customers
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .drop("rank", "timestamp_ingestion") 
)

salvar_silver(df_dim_consumidores, "dim_consumidores")

## 4 — silver.fat_pedidos
Origem: `bronze.tb_orders`

Regras:
- Tradução de status do pedido de inglês para português
- Colunas derivadas de tempo de entrega
- Indicador de entrega no prazo

In [0]:
df_orders_raw = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.tb_orders")

status_map = {
    "delivered"   : "entregue",
    "canceled"    : "cancelado",
    "shipped"     : "enviado",
    "processing"  : "processando",
    "invoiced"    : "faturado",
    "unavailable" : "indisponível",
    "created"     : "criado",
    "approved"    : "aprovado",
}

status_expr = F.col("order_status")
for en, pt in status_map.items():
    status_expr = F.when(F.col("order_status") == en, pt).otherwise(status_expr)

df_fat_pedidos = (
    df_orders_raw
    .withColumn("status", status_expr)
    .withColumn("data_compra",           F.to_timestamp("order_purchase_timestamp"))
    .withColumn("data_aprovacao",        F.to_timestamp("order_approved_at"))
    .withColumn("data_entrega_carrier",  F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("data_entrega_real",     F.to_timestamp("order_delivered_customer_date"))
    .withColumn("data_entrega_estimada", F.to_timestamp("order_estimated_delivery_date"))
    .withColumn(
        "tempo_entrega_dias",
        F.datediff(F.col("data_entrega_real"), F.col("data_compra"))
    )
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff(F.col("data_entrega_estimada"), F.col("data_compra"))
    )
    .withColumn(
        "diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
    )
    .withColumn(
        "entrega_no_prazo",
        F.when(
            F.col("status") == "entregue",
            F.when(F.col("data_entrega_real") <= F.col("data_entrega_estimada"), "Sim").otherwise("Não")
        ).otherwise("Não Entregue")
    )
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),
        "status",
        "data_compra",
        "data_aprovacao",
        "data_entrega_carrier",
        "data_entrega_real",
        "data_entrega_estimada",
        "tempo_entrega_dias",
        "tempo_entrega_estimado_dias",
        "diferenca_entrega_dias",
        "entrega_no_prazo",
    )
)

salvar_silver(df_fat_pedidos, "fat_pedidos")

## 5 — silver.fat_itens_pedidos
Origem: `bronze.tb_order_items`

Regras: renomeação de colunas para português com tipagem correta.

In [0]:
df_order_items_raw = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.tb_order_items")

df_fat_itens_pedidos = df_order_items_raw.select(
    F.col("order_id").alias("id_pedido"),
    F.col("order_item_id").cast(IntegerType()).alias("id_item"),
    F.col("product_id").alias("id_produto"),
    F.col("seller_id").alias("id_vendedor"),
    F.to_timestamp("shipping_limit_date").alias("data_limite_envio"),
    F.col("price").cast(DoubleType()).alias("preco_BRL"),
    F.col("freight_value").cast(DoubleType()).alias("preco_frete"),
)

salvar_silver(df_fat_itens_pedidos, "fat_itens_pedidos")

## 6 — silver.fat_pagamentos_pedidos
Origem: `bronze.tb_order_payments`

Regras: tradução do tipo de pagamento de inglês para português.

In [0]:
df_payments_raw = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.tb_order_payments")

pagamento_map = {
    "credit_card" : "Cartão de Crédito",
    "boleto"      : "Boleto",
    "voucher"     : "Voucher",
    "debit_card"  : "Cartão de Débito",
    "not_defined" : "Não Definido",
}

pagamento_expr = F.col("payment_type")
for en, pt in pagamento_map.items():
    pagamento_expr = F.when(F.col("payment_type") == en, pt).otherwise(pagamento_expr)

df_fat_pagamentos = df_payments_raw.select(
    F.col("order_id").alias("id_pedido"),
    F.col("payment_sequential").cast(IntegerType()).alias("sequencial_pagamento"),
    pagamento_expr.alias("tipo_pagamento"),
    F.col("payment_installments").cast(IntegerType()).alias("parcelas"),
    F.col("payment_value").cast(DoubleType()).alias("valor_pagamento"),
)

salvar_silver(df_fat_pagamentos, "fat_pagamentos_pedidos")

## 7 — silver.fat_avaliacoes_pedidos
Origem: `bronze.tb_order_reviews`

Regras:
- `try_to_timestamp` para tolerância a falhas em colunas de data
- Remove registros com `id_pedido` nulo ou datas de comentário no futuro
- Preenche títulos e comentários vazios com strings padrão

In [0]:
df_reviews_raw = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.tb_order_reviews")

df_fat_avaliacoes = (
    df_reviews_raw
    .withColumn("data_criacao_avaliacao",  F.try_to_timestamp(F.col("review_creation_date")))
    .withColumn("data_resposta_avaliacao", F.try_to_timestamp(F.col("review_answer_timestamp")))
    .filter(F.col("order_id").isNotNull())
    .filter(
        F.col("data_criacao_avaliacao").isNull() |
        (F.col("data_criacao_avaliacao") <= F.current_timestamp())
    )
    .withColumn(
        "titulo_avaliacao",
        F.when(
            F.col("review_comment_title").isNull() | (F.trim(F.col("review_comment_title")) == ""),
            "Sem título"
        ).otherwise(F.col("review_comment_title"))
    )
    .withColumn(
        "comentario_avaliacao",
        F.when(
            F.col("review_comment_message").isNull() | (F.trim(F.col("review_comment_message")) == ""),
            "Sem comentário"
        ).otherwise(F.col("review_comment_message"))
    )
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        F.expr("try_cast(review_score AS INT)").alias("nota_avaliacao"),
        "titulo_avaliacao",
        "comentario_avaliacao",
        "data_criacao_avaliacao",
        "data_resposta_avaliacao",
    )
    .filter(F.col("nota_avaliacao").isNotNull())
)

salvar_silver(df_fat_avaliacoes, "fat_avaliacoes_pedidos")


## 8 — silver.dim_produtos
Origem: `bronze.tb_products`

Regras:
- Renomeação de colunas para português
- Deduplicação Sênior com Window Function pelo `timestamp_ingestion`

In [0]:
df_products_raw = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.tb_products")

df_products_renamed = df_products_raw.select(
    F.col("product_id").alias("id_produto"),
    F.col("product_name").alias("nome_produto"),
    F.col("product_category_name").alias("categoria_produto"),
    F.col("product_name_lenght").cast(IntegerType()).alias("tamanho_nome_produto"),
    F.col("product_description_lenght").cast(IntegerType()).alias("tamanho_descricao_produto"),
    F.col("product_photos_qty").cast(IntegerType()).alias("quantidade_fotos"),
    F.col("product_weight_g").cast(DoubleType()).alias("peso_produto_gramas"),
    F.col("product_length_cm").cast(DoubleType()).alias("comprimento_centimetros"),
    F.col("product_height_cm").cast(DoubleType()).alias("altura_centimetros"),
    F.col("product_width_cm").cast(DoubleType()).alias("largura_centimetros"),
    F.col("timestamp_ingestion")
)

window_dedup_prod = Window.partitionBy("id_produto").orderBy(F.desc("timestamp_ingestion"))

df_dim_produtos = (
    df_products_renamed
    .withColumn("rank", F.row_number().over(window_dedup_prod))
    .filter(F.col("rank") == 1)
    .drop("rank", "timestamp_ingestion")
)

salvar_silver(df_dim_produtos, "dim_produtos")

## 9 — silver.dim_vendedores
Origem: `bronze.tb_sellers`

Regras:
- Renomeação de colunas para português
- Cidade e estado em UPPER CASE
- Deduplicação Sênior com Window Function pelo `timestamp_ingestion`

In [0]:
df_sellers_raw = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.tb_sellers")

df_sellers_renamed = df_sellers_raw.select(
    F.col("seller_id").alias("id_vendedor"),
    F.col("seller_name").alias("nome_vendedor"),
    F.col("seller_zip_code_prefix").cast(IntegerType()).alias("prefixo_cep"),
    F.upper(F.col("seller_city")).alias("cidade"),   
    F.upper(F.col("seller_state")).alias("estado"), 
    F.col("timestamp_ingestion")
)

window_dedup_sell = Window.partitionBy("id_vendedor").orderBy(F.desc("timestamp_ingestion"))

df_dim_vendedores = (
    df_sellers_renamed
    .withColumn("rank", F.row_number().over(window_dedup_sell))
    .filter(F.col("rank") == 1)
    .drop("rank", "timestamp_ingestion")
)

salvar_silver(df_dim_vendedores, "dim_vendedores")

## 10 — silver.dim_categoria_produtos_traducao
Origem: `bronze.tb_product_category_name_translation`

Regras: renomeação simples das colunas de nome de categoria para português e inglês.

In [0]:
df_cat_raw = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.tb_product_category_name_translation")

df_dim_categoria = df_cat_raw.select(
    F.col("product_category_name").alias("nome_produto_pt"),
    F.col("product_category_name_english").alias("nome_produto_en"),
)

salvar_silver(df_dim_categoria, "dim_categoria_produtos_traducao")

## 11 — silver.dim_cotacao_dolar
Origem: `bronze.tb_cotacao_dolar`

Regras:
- A API PTAX não retorna cotações para finais de semana e feriados
- Criamos um calendário contínuo cobrindo todos os dias do período
- Para dias sem cotação, usamos `last(ignorenulls=True)` sobre uma Window Function
  ordenada por data, preenchendo automaticamente com a cotação de fechamento
  do dia útil anterior (tipicamente a sexta-feira)

In [0]:
df_cotacao_raw = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.tb_cotacao_dolar")

df_cotacao_prep = df_cotacao_raw.select(
    F.to_date(F.to_timestamp(F.col("dataHoraCotacao"))).alias("data_cotacao"),
    F.col("cotacaoCompra").cast(DoubleType()).alias("cotacao_dolar"),
)

row_min_max = df_cotacao_prep.agg(
    F.min("data_cotacao").alias("data_min"),
    F.max("data_cotacao").alias("data_max")
).collect()[0]

data_min = row_min_max["data_min"]
data_max = row_min_max["data_max"]

print(f"Período da cotação: {data_min} → {data_max}")

df_calendario = spark.sql(f"""
    SELECT explode(sequence(
        DATE '{data_min}',
        DATE '{data_max}',
        INTERVAL 1 DAY
    )) AS data_cotacao
""")

df_joined = df_calendario.join(df_cotacao_prep, on="data_cotacao", how="left")

window_forward_fill = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_dim_cotacao = (
    df_joined
    .withColumn(
        "cotacao_dolar",
        F.last(F.col("cotacao_dolar"), ignorenulls=True).over(window_forward_fill)
    )
    .select("data_cotacao", "cotacao_dolar")
)

salvar_silver(df_dim_cotacao, "dim_cotacao_dolar")

## 12 — silver.fat_pedido_total
Tabela fato consolidada: join entre pedidos, pagamentos e cotação do dólar.

Contém: `id_pedido`, `id_consumidor`, `status`, `valor_total_pago_brl`, `valor_total_pago_usd`, `data_pedido`.

Os valores financeiros são arredondados a exatamente 2 casas decimais.

In [0]:
df_pedidos    = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.fat_pedidos")
df_pagamentos = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.fat_pagamentos_pedidos")
df_cotacao    = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.dim_cotacao_dolar")

df_pagamentos_agg = (
    df_pagamentos
    .groupBy("id_pedido")
    .agg(F.sum("valor_pagamento").alias("valor_total_pago_brl"))
)

df_pedido_total = (
    df_pedidos
    .join(df_pagamentos_agg, on="id_pedido", how="left")
    .join(
        df_cotacao,
        on=F.to_date(F.col("data_compra")) == F.col("data_cotacao"),
        how="left"
    )
    .withColumn(
        "valor_total_pago_usd",
        F.round(F.col("valor_total_pago_brl") / F.col("cotacao_dolar"), 2)
    )
    .withColumn("valor_total_pago_brl", F.round(F.col("valor_total_pago_brl"), 2))
    .select(
        "id_pedido",
        "id_consumidor",
        "status",
        "valor_total_pago_brl",
        "valor_total_pago_usd",
        F.col("data_compra").alias("data_pedido"),
    )
)

salvar_silver(df_pedido_total, "fat_pedido_total")

## 13 — Otimização Física (Delta OPTIMIZE + ZORDER)
Executa manutenção do Delta Lake nas principais tabelas fato.
O ZORDER reorganiza fisicamente os dados pelos campos mais filtrados,
acelerando drasticamente as consultas analíticas.

In [0]:
tabelas_optimize = [
    ("fat_pedidos",       "id_pedido, data_compra"),
    ("fat_pedido_total",  "id_pedido, data_pedido"),
    ("fat_itens_pedidos", "id_pedido"),
]

for tabela, zcols in tabelas_optimize:
    print(f"Otimizando {tabela} com ZORDER BY ({zcols})...")
    spark.sql(f"""
        OPTIMIZE {CATALOG}.{SCHEMA_SILVER}.{tabela}
        ZORDER BY ({zcols})
    """)
    print(f"  ✅ {tabela} otimizada")

## 14 — Validação Final

In [0]:
print("=== Tabelas na camada Silver ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_SILVER}"))